# Validación del histórico NYC 311 (2022–2025)

**Proyecto:** Despliegue de Soluciones Analíticas — NYC 311  
**Actividad:** Preparación y validación de datos para DSAP-34

## Objetivo

Verificar la estructura, cobertura temporal y consistencia del conjunto
de solicitudes NYC 311 utilizado para construir la variable objetivo
y las variables predictoras del modelo de demanda semanal.

La validación se realiza sobre el archivo Parquet agregado y se
contrasta con los resultados obtenidos desde la API oficial.

In [2]:
from pathlib import Path
import pandas as pd

raiz = Path.cwd()

if raiz.name == "notebooks":
    raiz = raiz.parent

ruta_parquet = raiz / "data/raw/nyc311_api_s3_nuevo.parquet"

df = pd.read_parquet(ruta_parquet)
df["dia"] = pd.to_datetime(df["dia"])

print("Fuente de datos:", ruta_parquet.name)
print("Dimensiones:", df.shape)
print("Total de solicitudes:", df["n"].sum())

display(df.head())
display(df.isna().sum().to_frame("Nulos"))

Fuente de datos: nyc311_api_s3_nuevo.parquet
Dimensiones: (652697, 4)
Total de solicitudes: 13506487


,borough,dia,complaint_type,n
0,BRONX,2022-01-01,Abandoned Vehicle,3
1,BRONX,2022-01-01,Animal-Abuse,2
2,BRONX,2022-01-01,Animal in a Park,1
3,BRONX,2022-01-01,APPLIANCE,5
4,BRONX,2022-01-01,Bike/Roller/Skate Chronic,1


,Nulos
borough,0
dia,0
complaint_type,0
n,0


In [3]:
# Validación de cobertura temporal y solicitudes
df["dia"] = pd.to_datetime(df["dia"])

print("Fecha mínima:", df["dia"].min())
print("Fecha máxima:", df["dia"].max())
print("Total de solicitudes:", f'{df["n"].sum():,}')

print("\nSolicitudes por año:")
display(
    df.groupby(df["dia"].dt.year)["n"]
    .sum()
    .to_frame("solicitudes")
)

print("\nSolicitudes por borough:")
display(
    df.groupby("borough")["n"]
    .sum()
    .sort_values(ascending=False)
    .to_frame("solicitudes")
)

print("\nTipos de solicitud distintos:", df["complaint_type"].nunique())

Fecha mínima: 2022-01-01 00:00:00
Fecha máxima: 2025-12-31 00:00:00
Total de solicitudes: 13,506,487

Solicitudes por año:


,solicitudes
dia,
2022,3169960
2023,3224723
2024,3456770
2025,3655034



Solicitudes por borough:


,solicitudes
borough,
BROOKLYN,4050079
QUEENS,3211377
BRONX,2936122
MANHATTAN,2775157
STATEN ISLAND,518683
Unspecified,15069



Tipos de solicitud distintos: 239


## Construcción de la demanda semanal

Se agrupan las solicitudes NYC 311 por semana y borough, considerando
los cinco distritos de Nueva York.

Se excluyen los registros clasificados como "Unspecified", porque
no corresponden a uno de los cinco boroughs del alcance del proyecto.

Esta agregación será la base para construir la variable objetivo y
las variables históricas del modelo.

In [4]:
# Construcción de la demanda semanal por borough

boroughs = [
    "BRONX",
    "BROOKLYN",
    "MANHATTAN",
    "QUEENS",
    "STATEN ISLAND"
]

df_modelo = df[df["borough"].isin(boroughs)].copy()

# Semanas que comienzan los lunes
df_modelo["semana"] = (
    df_modelo["dia"]
    - pd.to_timedelta(df_modelo["dia"].dt.weekday, unit="D")
).dt.normalize()

demanda_semanal = (
    df_modelo.groupby(["borough", "semana"], as_index=False)["n"]
    .sum()
    .rename(columns={"n": "total_solicitudes"})
    .sort_values(["borough", "semana"])
    .reset_index(drop=True)
)

print("Observaciones semanales:", len(demanda_semanal))
print("Boroughs:", demanda_semanal["borough"].nunique())
print("Primera semana:", demanda_semanal["semana"].min())
print("Última semana:", demanda_semanal["semana"].max())

display(demanda_semanal.head(10))

Observaciones semanales: 1050
Boroughs: 5
Primera semana: 2021-12-27 00:00:00
Última semana: 2025-12-29 00:00:00


,borough,semana,total_solicitudes
0,BRONX,2021-12-27,2256
1,BRONX,2022-01-03,12056
2,BRONX,2022-01-10,16275
3,BRONX,2022-01-17,14508
4,BRONX,2022-01-24,14915
5,BRONX,2022-01-31,14801
6,BRONX,2022-02-07,18860
7,BRONX,2022-02-14,14380
8,BRONX,2022-02-21,11279
9,BRONX,2022-02-28,13147


## Selección de semanas completas

Se excluyen las semanas parciales al inicio y al final del período
2022–2025 para evitar que la demanda incompleta afecte los rezagos,
los promedios móviles y la clasificación de la variable objetivo.

El conjunto final conserva 208 semanas por borough.

In [5]:
# Conservar únicamente semanas completas (lunes a domingo)

demanda_semanal = demanda_semanal[
    (demanda_semanal["semana"] >= "2022-01-03") &
    (demanda_semanal["semana"] <= "2025-12-22")
].copy().reset_index(drop=True)

# Validaciones
print("Observaciones finales:", len(demanda_semanal))
print("Primera semana:", demanda_semanal["semana"].min())
print("Última semana:", demanda_semanal["semana"].max())

print("\nSemanas por borough:")
display(demanda_semanal.groupby("borough").size().to_frame("semanas"))

print("Total de solicitudes en semanas completas:",
      f'{demanda_semanal["total_solicitudes"].sum():,}')

assert len(demanda_semanal) == 1040
assert demanda_semanal.groupby("borough").size().eq(208).all()

print("\nValidación completada correctamente.")

Observaciones finales: 1040
Primera semana: 2022-01-03 00:00:00
Última semana: 2025-12-22 00:00:00

Semanas por borough:


,semanas
borough,
BRONX,208
BROOKLYN,208
MANHATTAN,208
QUEENS,208
STATEN ISLAND,208


Total de solicitudes en semanas completas: 13,450,586

Validación completada correctamente.


## Construcción de variables predictoras — DSAP-32

Se construyen variables históricas y de calendario para anticipar
el nivel de demanda semanal de solicitudes NYC 311.

Los rezagos y las estadísticas móviles se calculan por borough,
utilizando exclusivamente información de semanas anteriores para
evitar fugas de información durante el entrenamiento.

Estas variables serán evaluadas posteriormente durante DSAP-34.

In [6]:
# Construcción de variables predictoras

datos_modelo = demanda_semanal.copy()
datos_modelo = datos_modelo.sort_values(
    ["borough", "semana"]
).reset_index(drop=True)

# Variables de calendario
datos_modelo["mes"] = datos_modelo["semana"].dt.month
datos_modelo["semana_anio"] = (
    datos_modelo["semana"].dt.isocalendar().week.astype(int)
)

# Rezagos por borough
grupo = datos_modelo.groupby("borough")["total_solicitudes"]

for rezago in [1, 2, 4, 52]:
    datos_modelo[f"lag_{rezago}"] = grupo.shift(rezago)

# Promedios y variabilidad histórica
for ventana in [4, 12, 52]:
    datos_modelo[f"media_movil_{ventana}"] = (
        datos_modelo.groupby("borough")["total_solicitudes"]
        .transform(
            lambda x: x.shift(1).rolling(
                window=ventana,
                min_periods=ventana
            ).mean()
        )
    )

datos_modelo["std_movil_4"] = (
    datos_modelo.groupby("borough")["total_solicitudes"]
    .transform(
        lambda x: x.shift(1).rolling(
            window=4,
            min_periods=4
        ).std()
    )
)

columnas_variables = [
    "borough", "semana", "total_solicitudes",
    "mes", "semana_anio",
    "lag_1", "lag_2", "lag_4", "lag_52",
    "media_movil_4", "media_movil_12",
    "media_movil_52", "std_movil_4"
]

display(datos_modelo[columnas_variables].tail(10))

print("Filas originales:", len(datos_modelo))
print("Filas con historial de 52 semanas:",
      datos_modelo["media_movil_52"].notna().sum())

,borough,semana,total_solicitudes,mes,semana_anio,lag_1,lag_2,lag_4,lag_52,media_movil_4,media_movil_12,media_movil_52,std_movil_4
1030,STATEN ISLAND,2025-10-20,2590,10,43,2544.0,2537.0,2482.0,2564.0,2547.75,2688.500000,2529.653846,60.257088
1031,STATEN ISLAND,2025-10-27,2761,10,44,2590.0,2544.0,2628.0,2583.0,2574.75,2656.833333,2530.153846,42.578359
1032,STATEN ISLAND,2025-11-03,2367,11,45,2761.0,2590.0,2537.0,2409.0,2608.00,2656.083333,2533.576923,104.674097
1033,STATEN ISLAND,2025-11-10,2300,11,46,2367.0,2761.0,2544.0,2288.0,2565.50,2634.416667,2532.769231,161.951639
1034,STATEN ISLAND,2025-11-17,2424,11,47,2300.0,2367.0,2590.0,2444.0,2504.50,2596.416667,2533.000000,211.210006
1035,STATEN ISLAND,2025-11-24,2072,11,48,2424.0,2300.0,2761.0,1638.0,2463.00,2566.916667,2532.615385,205.028453
1036,STATEN ISLAND,2025-12-01,2320,12,49,2072.0,2424.0,2367.0,2281.0,2290.75,2520.000000,2540.961538,154.387769
1037,STATEN ISLAND,2025-12-08,2595,12,50,2320.0,2072.0,2300.0,2146.0,2279.00,2472.416667,2541.711538,148.319475
1038,STATEN ISLAND,2025-12-15,3052,12,51,2595.0,2320.0,2424.0,1880.0,2352.75,2468.333333,2550.346154,218.826225
1039,STATEN ISLAND,2025-12-22,1978,12,52,3052.0,2595.0,2072.0,1887.0,2509.75,2515.833333,2572.884615,419.893935


Filas originales: 1040
Filas con historial de 52 semanas: 780


## Construcción de la variable objetivo — DSAP-30

Se clasifica la demanda semanal de NYC 311 en tres categorías:
Baja, Media y Alta.

Para cada borough se calcula la demanda relativa respecto al
promedio de las 52 semanas anteriores.

Los umbrales P33 y P67 se estiman utilizando solamente los datos
de 2023–2024 y se mantienen constantes para evaluar 2025.

Así evitamos utilizar información del período de prueba para
definir las categorías.

In [7]:
import numpy as np

# Trabajar con semanas que ya tienen historial suficiente
datos_modelo = datos_modelo.dropna(
    subset=["media_movil_52"]
).copy()

# Demanda relativa
datos_modelo["demanda_relativa"] = (
    datos_modelo["total_solicitudes"] /
    datos_modelo["media_movil_52"]
)

# Umbrales por borough, calculados únicamente con 2023–2024
entrenamiento = datos_modelo[
    datos_modelo["semana"].dt.year.isin([2023, 2024])
]

umbrales = (
    entrenamiento.groupby("borough")["demanda_relativa"]
    .quantile([0.33, 0.67])
    .unstack()
    .rename(columns={0.33: "P33", 0.67: "P67"})
)

# Incorporar los umbrales a cada borough
datos_modelo = datos_modelo.merge(
    umbrales, on="borough", how="left"
)

# Clasificación multiclase
datos_modelo["nivel_demanda"] = np.select(
    [
        datos_modelo["demanda_relativa"] < datos_modelo["P33"],
        datos_modelo["demanda_relativa"] <= datos_modelo["P67"]
    ],
    ["Baja", "Media"],
    default="Alta"
)

print("Observaciones disponibles:", len(datos_modelo))
print("\nUmbrales por borough:")
display(umbrales.round(3))

print("\nDistribución de clases:")
display(
    pd.crosstab(
        datos_modelo["borough"],
        datos_modelo["nivel_demanda"]
    )
)

Observaciones disponibles: 780

Umbrales por borough:


,P33,P67
borough,,
BRONX,0.937,1.069
BROOKLYN,0.992,1.077
MANHATTAN,0.982,1.078
QUEENS,0.988,1.087
STATEN ISLAND,0.919,1.042



Distribución de clases:


nivel_demanda,Alta,Baja,Media
borough,,,
BRONX,48,66,42
BROOKLYN,40,50,66
MANHATTAN,44,54,58
QUEENS,53,52,51
STATEN ISLAND,60,40,56


In [8]:
from pathlib import Path

ruta_salida = Path("../data/processed/dataset_modelado_semanal.parquet")

if not Path("../data").exists():
    ruta_salida = Path("data/processed/dataset_modelado_semanal.parquet")

ruta_salida.parent.mkdir(parents=True, exist_ok=True)

datos_modelo.to_parquet(ruta_salida, index=False)

print("Dataset guardado en:", ruta_salida.resolve())
print("Observaciones:", len(datos_modelo))
print("Columnas:", len(datos_modelo.columns))

Dataset guardado en: C:\Users\georg\Documents\Uniandes\DSA\dsa-proyecto\data\processed\dataset_modelado_semanal.parquet
Observaciones: 780
Columnas: 17


In [9]:
# dimensiones finales

print("Dimensiones finales:", datos_modelo.shape)
print("\nDistribución por año:")
print(datos_modelo["semana"].dt.year.value_counts().sort_index())

print("\nDistribución de la variable objetivo:")
print(datos_modelo["nivel_demanda"].value_counts())

print("\nValores nulos:")
print(datos_modelo.isna().sum().sum())

Dimensiones finales: (780, 17)

Distribución por año:
semana
2023    260
2024    265
2025    255
Name: count, dtype: int64

Distribución de la variable objetivo:
nivel_demanda
Media    273
Baja     262
Alta     245
Name: count, dtype: int64

Valores nulos:
0
